# Chatbot con MCP, Ollama y Gradio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/3-chatbot-con-mcp-y-gradio.ipynb)

Este notebook es un ejemplo avanzado y opcional de la sesión. No introduce conceptos nuevos: simplemente toma el agente con herramientas MCP del notebook anterior y le monta encima una interfaz conversacional con Gradio, siguiendo el estilo visto antes en la sesión de RAG con LangChain. La idea es cerrar la unidad mostrando cómo pasar de una integración técnica a una experiencia de usuario lista para demostración.

### Referencias
- [Gradio](https://www.gradio.app/)
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Ollama](https://ollama.com/)


In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages


In [ ]:
!test '{IN_COLAB}' = 'True' && pip install mcp langchain langchain-core langchain-ollama langchain-mcp-adapters langgraph httpx gradio ollama colab-xterm


### Cargando a Ollama

Usaremos el mismo modelo local y la misma idea de tools de los notebooks previos para que lo único nuevo aquí sea la interfaz.


In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

En Colab recuerda arrancar `ollama serve` desde la terminal embebida si todavía no está corriendo. En local basta con tener el servicio levantado de antemano.

In [ ]:
%load_ext colabxterm
%xterm


Mantendremos `llama3.2:3b` para la demostración.


In [ ]:
!ollama pull llama3.2:3b


## Reutilizamos los mismos servidores MCP

Para que el notebook sea autosuficiente, volvemos a escribir los dos servidores mínimos: uno para la calculadora y otro para el clima actual. Son exactamente el mismo tipo de herramientas del notebook anterior; lo único que cambia después es cómo las exponemos a la persona usuaria final.

In [ ]:
from pathlib import Path

SERVERS_DIR = Path.cwd() / 'mcp_servers'
SERVERS_DIR.mkdir(exist_ok=True)
SERVERS_DIR


In [ ]:
calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
calculator_server.write_text('from mcp.server.fastmcp import FastMCP\nimport ast\nimport operator as op\n\nmcp = FastMCP("CalculadoraMCP")\n\nALLOWED_BIN_OPS = {\n    ast.Add: op.add,\n    ast.Sub: op.sub,\n    ast.Mult: op.mul,\n    ast.Div: op.truediv,\n    ast.Pow: op.pow,\n    ast.Mod: op.mod,\n}\nALLOWED_UNARY_OPS = {\n    ast.UAdd: op.pos,\n    ast.USub: op.neg,\n}\n\ndef evaluate_expression(expression: str):\n    def _eval(node):\n        if isinstance(node, ast.Expression):\n            return _eval(node.body)\n        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):\n            return node.value\n        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BIN_OPS:\n            return ALLOWED_BIN_OPS[type(node.op)](_eval(node.left), _eval(node.right))\n        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARY_OPS:\n            return ALLOWED_UNARY_OPS[type(node.op)](_eval(node.operand))\n        raise ValueError("La expresión contiene operadores no soportados.")\n\n    return _eval(ast.parse(expression, mode="eval"))\n\n@mcp.tool()\ndef calculadora(expression: str) -> str:\n    """Evalúa una expresión aritmética segura con +, -, *, /, %, ** y paréntesis."""\n    result = evaluate_expression(expression)\n    return f"Resultado exacto: {result}"\n\nif __name__ == "__main__":\n    mcp.run(transport="stdio")\n', encoding='utf-8')
print(calculator_server.read_text(encoding='utf-8'))


In [ ]:
weather_server = SERVERS_DIR / 'weather_mcp_server.py'
weather_server.write_text('from mcp.server.fastmcp import FastMCP\nimport httpx\n\nmcp = FastMCP("ClimaMCP")\n\n@mcp.tool()\ndef clima_actual(city: str) -> str:\n    """Consulta el clima actual de una ciudad usando Open-Meteo."""\n    geocode_response = httpx.get(\n        "https://geocoding-api.open-meteo.com/v1/search",\n        params={"name": city, "count": 1, "language": "es", "format": "json"},\n        timeout=30.0,\n    )\n    geocode_response.raise_for_status()\n    geocode_data = geocode_response.json()\n    if not geocode_data.get("results"):\n        return f"No encontré información para la ciudad: {city}"\n\n    location = geocode_data["results"][0]\n    weather_response = httpx.get(\n        "https://api.open-meteo.com/v1/forecast",\n        params={\n            "latitude": location["latitude"],\n            "longitude": location["longitude"],\n            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",\n            "timezone": "auto",\n        },\n        timeout=30.0,\n    )\n    weather_response.raise_for_status()\n    weather_data = weather_response.json()["current"]\n\n    return (\n        f"Clima actual en {location[\'name\']}, {location.get(\'country\', \'\')}: "\n        f"temperatura {weather_data[\'temperature_2m\']}°C, "\n        f"humedad {weather_data[\'relative_humidity_2m\']}%, "\n        f"viento {weather_data[\'wind_speed_10m\']} km/h, "\n        f"weather_code {weather_data[\'weather_code\']}."\n    )\n\nif __name__ == "__main__":\n    mcp.run(transport="stdio")\n', encoding='utf-8')
print(weather_server.read_text(encoding='utf-8'))


## Construimos el agente consumidor de herramientas MCP

Igual que antes, el agente no conoce directamente las funciones de Python; descubre las capacidades a través de los servidores MCP y decide cuándo invocarlas.

In [ ]:
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(calculator_server)],
        },
        'clima': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(weather_server)],
        },
    }
)

mcp_tools = await client.get_tools()
agent_mcp = create_react_agent(model=llm, tools=mcp_tools)


## Del agente a la interfaz conversacional

Aquí reaparece una idea de la sesión de RAG: Gradio maneja un historial simple para la interfaz, mientras nosotros reconstruimos a partir de él la secuencia de mensajes que el agente necesita. No agregamos memoria nueva al modelo; simplemente transformamos el historial visual en mensajes de conversación.

In [ ]:
import gradio as gr

def history_to_messages(question, chat_history):
    messages = []
    for human, assistant in chat_history:
        messages.append(('user', human))
        messages.append(('assistant', assistant))
    messages.append(('user', question))
    return messages

def normalize_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get('text') or item.get('content') or str(item))
            else:
                parts.append(str(item))
        return '\n'.join(parts)
    return str(content)

async def respond(question, chat_history):
    chat_history = chat_history or []
    result = await agent_mcp.ainvoke({'messages': history_to_messages(question, chat_history)})
    answer = normalize_content(result['messages'][-1].content)
    chat_history.append((question, answer))
    return '', chat_history

def reset_chat():
    return '', []


## Lanzando la interfaz de chat

Esta versión es opcional precisamente porque ya no enseña un concepto nuevo de NLP o de MCP; enseña cómo empaquetar el agente en una demo usable. Es útil para estudiantes que quieran presentar el flujo completo de extremo a extremo.

In [ ]:
with gr.Blocks() as gr_blocks:
    gr.Markdown('## Chat con herramientas MCP')
    chatbot = gr.Chatbot(label='Historial')
    msg = gr.Textbox(
        label='¿Qué quieres preguntar?',
        placeholder='Ejemplo: ¿Cuál es la temperatura actual en Cali?'
    )
    clear = gr.Button('Limpiar')

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(reset_chat, None, [msg, chatbot], queue=False)

gr_blocks.launch(inline=False)


In [ ]:
gr_blocks.close()


## Conclusiones

- Este notebook solo agrega una capa de presentación; el corazón sigue siendo el mismo agente con herramientas MCP.
- Gradio permite convertir el experimento técnico en una demo conversacional muy rápidamente.
- Como material opcional, ayuda a cerrar la sesión con una visión más aplicada sin sobrecargar el flujo conceptual principal.
